<a href="https://colab.research.google.com/github/semal-1820/AI-VFX-Agent-Pipeline/blob/main/visualfx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers diffusers accelerate transformers[torch] diffusers[torch] opencv-python pillow

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import SamModel, SamProcessor
from diffusers import StableDiffusionInpaintPipeline

class AIVFXPipeline:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")

        print("Loading Object Detection Model (OWLv2)...")
        self.det_processor = AutoProcessor.from_pretrained("google/owlv2-base-patch16")
        self.det_model = AutoModelForZeroShotObjectDetection.from_pretrained("google/owlv2-base-patch16").to(self.device)

        print("Loading Segmentation Model (SAM)...")
        self.sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
        self.sam_model = SamModel.from_pretrained("facebook/sam-vit-base").to(self.device)

        print("Loading Inpainting Model (Stable Diffusion)...")
        self.inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "runwayml/stable-diffusion-inpainting",
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        ).to(self.device)

    def detect_object(self, image: Image.Image, text_queries: list):
        """Locates the bounding box of the target text prompt with a relaxed threshold."""
        inputs = self.det_processor(text=text_queries, images=image, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.det_model(**inputs)

        target_sizes = torch.tensor([image.size[::-1]]).to(self.device)

        # THE FIX: We completely removed the unexpected 'text_queries' argument here!
        results = self.det_processor.post_process_grounded_object_detection(
            outputs=outputs, threshold=0.05, target_sizes=target_sizes
        )

        # SAFETY CHECK: If no boxes found, use a fallback box covering a default region
        if len(results[0]["boxes"]) > 0:
            best_box = results[0]["boxes"][0].cpu().numpy().astype(int).tolist()
            return best_box, False
        else:
            print(f"⚠️ Warning: Direct match for '{text_queries[0]}' not found. Applying fallback region.")
            w, h = image.size
            fallback_box = [0, 0, w, h // 2]
            return fallback_box, True

    def generate_mask(self, image: Image.Image, bbox: list, is_fallback: bool):
        """Uses SAM to turn a bounding box into a detailed binary mask."""
        if is_fallback:
            w, h = image.size
            mask_np = np.zeros((h, w), dtype=np.uint8)
            mask_np[0:h//2, 0:w] = 255
            return Image.fromarray(mask_np)

        input_boxes = [[bbox]]
        inputs = self.sam_processor(image, input_boxes=input_boxes, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.sam_model(**inputs)

        masks = self.sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu()
        )

        mask_np = masks[0][0][0].numpy().astype(np.uint8) * 255
        return Image.fromarray(mask_np)

    def inpaint_scene(self, original_image: Image.Image, mask_image: Image.Image, prompt: str):
        """Inpaints the masked region with the new generated background/object."""
        w, h = original_image.size
        w, h = (w // 8) * 8, (h // 8) * 8

        img_resized = original_image.resize((w, h))
        mask_resized = mask_image.resize((w, h))

        output = self.inpaint_pipe(
            prompt=prompt,
            image=img_resized,
            mask_image=mask_resized,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

        return output.resize((w, h))

    def run_vfx_pipeline(self, image_path: str, target_to_replace: str, new_prompt: str):
        """Orchestrates the entire end-to-end pipeline execution safely."""
        print("\n--- Starting Pipeline Execution ---")
        original_image = Image.open(image_path).convert("RGB")

        print(f"Step 1: Locating '{target_to_replace}' in the frame...")
        bbox, is_fallback = self.detect_object(original_image, text_queries=[target_to_replace])
        print(f"Target box coordinates: {bbox}")

        print("Step 2: Generating mask...")
        mask_image = self.generate_mask(original_image, bbox, is_fallback)

        print(f"Step 3: Replacing masked area with '{new_prompt}'...")
        final_output = self.inpaint_scene(original_image, mask_image, prompt=new_prompt)

        print("--- Process Complete! ---")
        return mask_image, final_output

In [ ]:
vfx_agent = AIVFXPipeline()

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

# Wrapper function to connect Gradio to your pipeline
def process_image_ui(input_image_path, target_object, generative_prompt):
    print(f"UI Request received: Replace '{target_object}' with '{generative_prompt}'")

    # Run the pipeline (vfx_agent is already loaded in Cell 3)
    mask, result = vfx_agent.run_vfx_pipeline(
        image_path=input_image_path,
        target_to_replace=target_object,
        new_prompt=generative_prompt
    )

    return mask, result

# Build the Web Interface
with gr.Blocks(theme=gr.themes.Soft()) as vfx_interface:
    gr.Markdown("# 🎬 AI VFX Agent Pipeline")
    gr.Markdown("Upload an image, specify the object you want to mask, and describe the new visual effect.")

    with gr.Row():
        # Left Column: User Inputs
        with gr.Column():
            img_input = gr.Image(type="filepath", label="1. Upload Original Image")
            target_input = gr.Textbox(
                label="2. Object to Replace",
                placeholder="e.g., 'sky', 'person', 'car'"
            )
            prompt_input = gr.Textbox(
                label="3. New VFX Prompt",
                placeholder="e.g., 'a vibrant cosmic sci-fi nebula'"
            )
            submit_btn = gr.Button("Generate VFX", variant="primary")

        # Right Column: AI Outputs
        with gr.Column():
            mask_output = gr.Image(label="Intermediate Step: SAM Mask")
            final_output = gr.Image(label="Final Output: Inpainted VFX")

    # Connect the button to the function
    submit_btn.click(
        fn=process_image_ui,
        inputs=[img_input, target_input, prompt_input],
        outputs=[mask_output, final_output]
    )

# Launch the app!
# share=True creates a temporary public link you can share in your documentation.
vfx_interface.launch(debug=True, share=True)